In [13]:
import pandas as pd
import numpy as np

preds = pd.read_csv("../models/predictions.csv")

preds["value"] = preds["value"] / 10

gk_preds = preds[preds["pos"] == "GK"]
def_preds = preds[preds["pos"] == "DEF"]
mid_preds = preds[preds["pos"] == "MID"]
fwd_preds = preds[preds["pos"] == "FWD"]

In [14]:
fg_picks = pd.read_csv("./data/fg_manager_picks.csv")
fg_picks[(fg_picks["gameweek"] == 20) & (fg_picks["manager"] == "loother")].shape

(22, 12)

## Base optimizer

For each gameweek from 20, picks the 11 players, the captain and the vice-captain with the highest total
predicted xP (`pred`) under the FantasyGo round-contest rules, independently of other gameweeks (unlimited
changes before each deadline):

* exactly 1 GK, 3 to 5 DEF, 2 to 5 MID and 1 to 3 FWD
* a budget of 80 for the 11, at each player's FPL price that gameweek
* at most 3 players from one club
* a captain (points x2) and a vice-captain (points x1.5), the multipliers in the FantasyGo manager data

Every player with a fixture is a candidate, also those who then did not play, so the team is picked as
it could be before the deadline. It is scored with the actual points (`total_points`) and compared with
the FantasyGo managers of the same gameweek.

In [15]:
import pulp

BUDGET = 80.0
XI_SIZE = 11
FORMATION = {
    "GK": (1, 1),
    "DEF": (3, 5),
    "MID": (2, 5),
    "FWD": (1, 3),
}  # (min, max) per position
MAX_PER_TEAM = 3
CAPTAIN, VICE = 2.0, 1.5  # points multipliers


def best_xi(gw_preds, objective="pred"):
    """The 11 players, captain and vice-captain with the highest total `objective`."""
    players = gw_preds.reset_index(drop=True)
    idx = players.index
    prob = pulp.LpProblem("xi", pulp.LpMaximize)
    pick = [pulp.LpVariable(f"pick_{i}", cat="Binary") for i in idx]
    captain = [pulp.LpVariable(f"captain_{i}", cat="Binary") for i in idx]
    vice = [pulp.LpVariable(f"vice_{i}", cat="Binary") for i in idx]

    prob += pulp.lpSum(
        players.at[i, objective]
        * (pick[i] + (CAPTAIN - 1) * captain[i] + (VICE - 1) * vice[i])
        for i in idx
    )
    prob += pulp.lpSum(pick) == XI_SIZE
    prob += pulp.lpSum(captain) == 1
    prob += pulp.lpSum(vice) == 1
    for i in idx:
        prob += (
            captain[i] + vice[i] <= pick[i]
        )  # both in the XI, and not the same player
    prob += pulp.lpSum(players.at[i, "value"] * pick[i] for i in idx) <= BUDGET
    for pos, (low, high) in FORMATION.items():
        n = pulp.lpSum(pick[i] for i in idx[players["pos"] == pos])
        prob += n >= low
        prob += n <= high
    for team in players["team"].unique():
        prob += (
            pulp.lpSum(pick[i] for i in idx[players["team"] == team]) <= MAX_PER_TEAM
        )

    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    assert pulp.LpStatus[prob.status] == "Optimal", pulp.LpStatus[prob.status]
    chosen = players.assign(
        captain=[c.value() > 0.5 for c in captain],
        vice=[v.value() > 0.5 for v in vice],
    )
    chosen = chosen[[v.value() > 0.5 for v in pick]]
    chosen["multiplier"] = np.where(
        chosen["captain"], CAPTAIN, np.where(chosen["vice"], VICE, 1.0)
    )
    order = {pos: n for n, pos in enumerate(FORMATION)}
    return chosen.sort_values(
        ["pos", "pred"], key=lambda s: s.map(order) if s.name == "pos" else -s
    )

In [16]:
fg_managers = pd.read_csv("./data/fg_managers.csv")
fg_by_gw = fg_managers.groupby("gameweek")["points"].agg(
    fg_median="median", fg_best="max"
)

teams, summary = [], []
for gw, gw_preds in preds[preds["round"] >= 20].groupby("round"):
    xi = best_xi(gw_preds)
    teams.append(xi)
    summary.append(
        {
            "round": gw,
            "formation": "-".join(
                str((xi["pos"] == p).sum()) for p in ["DEF", "MID", "FWD"]
            ),
            "cost": xi["value"].sum(),
            "pred": (xi["pred"] * xi["multiplier"]).sum(),
            "points": (xi["total_points"] * xi["multiplier"]).sum(),
            "did_not_play": int((~xi["played"]).sum()),
            "captain": xi.loc[xi["captain"], "name"].iloc[0],
            "vice": xi.loc[xi["vice"], "name"].iloc[0],
        }
    )

teams = pd.concat(teams, ignore_index=True)
summary = pd.DataFrame(summary).set_index("round").join(fg_by_gw).round(1)
summary["beats_median"] = summary["points"] > summary["fg_median"]
print(
    f"Gameweeks {summary.index.min()}-{summary.index.max()}: predicted {summary['pred'].sum():.0f}, "
    f"actual {summary['points'].sum():.0f} points ({summary['points'].mean():.1f} per gameweek); "
    f"FantasyGo median manager {summary['fg_median'].mean():.1f} per gameweek, "
    f"beaten in {summary['beats_median'].sum()} of {len(summary)} gameweeks"
)
summary

Gameweeks 20-38: predicted 1644, actual 1164 points (61.2 per gameweek); FantasyGo median manager 64.3 per gameweek, beaten in 10 of 19 gameweeks


,formation,cost,pred,points,did_not_play,captain,vice,fg_median,fg_best,beats_median
round,,,,,,,,,,
20,3-4-3,70.5,78.5,36.5,1,Trent Alexander-Arnold,Julián Álvarez,71.5,112.0,False
21,3-4-3,79.5,81.7,40.0,2,Julián Álvarez,Mohamed Salah,60.5,118.5,False
22,3-5-2,77.7,83.5,53.0,2,Bernardo Veiga de Carvalho e Silva,Diogo Teixeira da Silva,47.2,83.5,True
23,3-5-2,77.4,77.2,41.0,1,Diogo Teixeira da Silva,Conor Bradley,37.5,75.5,True
24,3-5-2,74.1,86.7,35.0,2,Kevin De Bruyne,Conor Bradley,35.5,54.5,False
25,3-4-3,77.4,87.1,71.5,1,Bukayo Saka,Darwin Núñez Ribeiro,71.5,112.0,False
26,3-4-3,79.5,93.0,83.5,0,Bukayo Saka,Phil Foden,56.0,84.0,True
27,3-4-3,78.5,90.1,87.0,0,Bukayo Saka,Gabriel dos Santos Magalhães,77.5,106.5,True
28,3-4-3,71.1,87.1,38.0,0,Kai Havertz,Phil Foden,62.0,99.5,False


## The optimizer among the FantasyGo managers

Each gameweek is one FantasyGo contest. The optimizer's team is ranked among its entries by points: its
rank is one more than the number of entries with more points, so it shares a rank with any it ties.
`prize` is what the entry at that rank won (a tie may have split it).

In [17]:
OPTIMIZER = "** optimizer **"


def standings(gw):
    """The gameweek's FantasyGo entries and the optimizer's team, best first."""
    board = fg_managers.loc[
        fg_managers["gameweek"] == gw, ["manager", "entry", "points", "prize"]
    ]
    board = board.sort_values("points", ascending=False, kind="stable").reset_index(
        drop=True
    )
    ours = summary.at[gw, "points"]
    rank = int((board["points"] > ours).sum()) + 1
    prize = board["prize"].iloc[rank - 1] if rank <= len(board) else 0.0
    us = pd.DataFrame(
        [{"manager": OPTIMIZER, "entry": None, "points": ours, "prize": prize}]
    )
    board = pd.concat(
        [board.iloc[: rank - 1], us, board.iloc[rank - 1 :]], ignore_index=True
    )
    board.insert(
        0, "rank", board["points"].rank(method="min", ascending=False).astype(int)
    )
    return board, rank


def around_optimizer(gw, above=5, below=5):
    """The optimizer's row in the gameweek's standings, with `above` entries above and `below` below."""
    board, rank = standings(gw)
    at = board.index[board["manager"] == OPTIMIZER][0]
    return board.iloc[max(0, at - above) : at + below + 1]


ranks = []
for gw in summary.index:
    board, rank = standings(gw)
    entries = len(board) - 1
    ranks.append(
        {
            "round": gw,
            "rank": rank,
            "entries": entries,
            "top_pct": 100 * rank / (entries + 1),
            "prize": board.loc[board["manager"] == OPTIMIZER, "prize"].iloc[0],
        }
    )
ranks = pd.DataFrame(ranks).set_index("round").round(1)
print(
    f"Median rank in the top {ranks['top_pct'].median():.0f}%; "
    f"in the prizes in {(ranks['prize'] > 0).sum()} of {len(ranks)} gameweeks, "
    f"winning {ranks['prize'].sum():,.2f} in total"
)
ranks

Median rank in the top 44%; in the prizes in 4 of 19 gameweeks, winning 1,278.80 in total


,rank,entries,top_pct,prize
round,,,,
20,309,317,97.2,0.0
21,259,273,94.5,0.0
22,82,284,28.8,0.0
23,102,261,38.9,0.0
24,142,269,52.6,0.0
25,158,317,49.7,0.0
26,2,247,0.8,1204.8
27,54,226,23.8,23.4
28,199,204,97.1,0.0


In [52]:
rankings = pd.DataFrame({})
for gw in summary.index:
    rank = ranks.at[gw, "rank"]

    managers = around_optimizer(gw)
    manager = managers[managers["manager"] == "** optimizer **"]
    # print(manager)
    rankings = pd.concat(
        [
            rankings,
            pd.DataFrame(
                [
                    {
                        "gw": gw,
                        "rank": rank,
                        "points": manager["points"].values[0],
                        "prize": manager.prize.values[0],
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    # rankings
    # print(f"Gameweek {gw}: rank {rank} of {ranks.at[gw, 'entries'] + 1}")
    # display(
    #     around_optimizer(gw)
    #     .style.hide(axis="index")
    #     .apply(
    #         lambda row: [
    #             "font-weight: bold" if row["manager"] == OPTIMIZER else "" for _ in row
    #         ],
    #         axis=1,
    #     )
    #     .format({"points": "{:.1f}", "prize": "{:,.2f}", "entry": "{:.0f}"}, na_rep="")
    # )

rankings

,gw,rank,points,prize
0,20,309,36.5,0.00
1,21,259,40.0,0.00
2,22,82,53.0,0.00
3,23,102,41.0,0.00
4,24,142,35.0,0.00
5,25,158,71.5,0.00
6,26,2,83.5,1204.85
7,27,54,87.0,23.35
8,28,199,38.0,0.00
9,29,100,25.0,0.00


In [19]:
# The team for one gameweek
teams[teams["round"] == 20][
    [
        "pos",
        "name",
        "team",
        "value",
        "pred",
        "total_points",
        "minutes",
        "captain",
        "vice",
    ]
]

,pos,name,team,value,pred,total_points,minutes,captain,vice
0,GK,Alisson Ramses Becker,Liverpool,5.6,4.889919,1,90.0,False,False
1,DEF,Trent Alexander-Arnold,Liverpool,8.4,8.202660,0,90.0,True,False
2,DEF,Virgil van Dijk,Liverpool,6.2,5.433331,1,90.0,False,False
3,DEF,Gabriel dos Santos Magalhães,Arsenal,5.0,5.173240,1,90.0,False,False
4,MID,Michael Olise,Crystal Palace,5.8,6.449128,15,90.0,False,False
5,MID,Bernardo Veiga de Carvalho e Silva,Man City,6.5,6.299120,3,90.0,False,False
6,MID,Mohammed Kudus,West Ham,6.9,6.099955,0,0.0,False,False
7,MID,Jack Grealish,Man City,7.2,6.019302,1,51.0,False,False
8,FWD,Julián Álvarez,Man City,6.9,6.464999,7,90.0,False,True
9,FWD,Chris Wood,Nott'm Forest,4.9,6.058169,2,90.0,False,False
